# publish_all_helpfiles

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/publish_all_helpfiles.md`


Notebook source link: [publish_all_helpfiles.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/publish_all_helpfiles.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "publish_all_helpfiles"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"publish_all_helpfiles: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"publish_all_helpfiles: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"publish_all_helpfiles: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"publish_all_helpfiles: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# publish_all_helpfiles: Python-side publish/audit checks for help artifacts.
import json
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import yaml


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve()]
    candidates.append(candidates[0].parent)
    candidates.append(candidates[1].parent)
    for root in candidates:
        if (root / "docs" / "help").exists() and (root / "parity").exists():
            return root
    return candidates[0]


def walk_targets(nodes):
    targets = []
    for node in nodes or []:
        target = str(node.get("target", "")).strip()
        if target:
            targets.append(target)
        targets.extend(walk_targets(node.get("children", [])))
    return targets


def target_exists(repo_root: Path, help_root: Path, target: str) -> bool:
    candidate = Path(target)
    candidates = []
    if candidate.is_absolute():
        candidates.append(candidate)
    else:
        candidates.append(help_root / candidate)
        candidates.append(repo_root / "docs" / candidate)
        candidates.append(repo_root / candidate)
    return any(path.exists() for path in candidates)


repo_root = resolve_repo_root()
help_root = repo_root / "docs" / "help"
example_root = help_root / "examples"

eval_code = True
expected_generator = "sphinx"

staging_dir = Path(tempfile.mkdtemp(prefix="nstat_help_stage_"))
output_dir = Path(tempfile.mkdtemp(prefix="nstat_help_output_"))
staging_help = staging_dir / "help"
shutil.copytree(help_root, staging_help, dirs_exist_ok=True)

for pattern in ("*.asv", "*.bak", "*.ipynb", "*~", "publish_all_helpfiles.*", "temp.*"):
    for path in staging_help.rglob(pattern):
        if path.is_file():
            path.unlink()

subprocess.run(
    [sys.executable, str(repo_root / "tools" / "docs" / "generate_help_pages.py")],
    cwd=repo_root,
    check=True,
)
shutil.copytree(help_root, output_dir / "help", dirs_exist_ok=True)

manifest_path = repo_root / "parity" / "example_mapping.yaml"
manifest = yaml.safe_load(manifest_path.read_text(encoding="utf-8")) or {}
topics = [str(row.get("matlab_topic")) for row in manifest.get("examples", []) if row.get("matlab_topic")]

missing_example_pages = []
for topic in topics:
    if not (example_root / f"{topic}.md").exists():
        missing_example_pages.append(topic)

helptoc_path = help_root / "helptoc.yml"
helptoc = yaml.safe_load(helptoc_path.read_text(encoding="utf-8")) or {}
targets = sorted(set(walk_targets(helptoc.get("toc", helptoc.get("entries", [])))))
missing_targets = [target for target in targets if not target_exists(repo_root, help_root, target)]

help_files = sorted(path for path in help_root.rglob("*") if path.is_file())
n_md = sum(1 for path in help_files if path.suffix.lower() == ".md")
n_html = sum(1 for path in help_files if path.suffix.lower() == ".html")

html_root = repo_root / "docs" / "_build" / "html"
html_files = list(html_root.rglob("*.html")) if html_root.exists() else []
generator_hits = 0
for path in html_files[:200]:
    raw = path.read_text(encoding="utf-8", errors="ignore").lower()
    if "meta name=\"generator\"" in raw and expected_generator in raw:
        generator_hits += 1

audit_path = repo_root / "tests" / "parity" / "fixtures" / "matlab_gold" / "publish_all_helpfiles_audit_gold.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
audit_alignment = str(audit.get("alignment_status", ""))

fig, axes = plt.subplots(2, 2, figsize=(10.0, 6.8))
axes[0, 0].bar(
    ["manifest topics", "missing pages"],
    [len(topics), len(missing_example_pages)],
    color=["tab:blue", "tab:red"],
)
axes[0, 0].set_title(f"{TOPIC}: example-page coverage")
axes[0, 0].set_ylabel("count")

axes[0, 1].bar(
    ["TOC targets", "missing targets"],
    [len(targets), len(missing_targets)],
    color=["tab:green", "tab:red"],
)
axes[0, 1].set_title("helptoc target validation")
axes[0, 1].set_ylabel("count")

axes[1, 0].bar(
    ["markdown files", "html files"],
    [n_md, n_html],
    color=["tab:cyan", "tab:orange"],
)
axes[1, 0].set_title("help artifact inventory")
axes[1, 0].set_ylabel("count")

axes[1, 1].bar(
    ["staged files", "generator hits"],
    [sum(1 for path in staging_help.rglob("*") if path.is_file()), generator_hits],
    color=["tab:purple", "tab:olive"],
)
axes[1, 1].set_title("stage/output quality checks")
axes[1, 1].set_ylabel("count")
plt.tight_layout()
plt.show()

shutil.rmtree(staging_dir, ignore_errors=True)
shutil.rmtree(output_dir, ignore_errors=True)

assert eval_code is True
assert len(topics) > 0
assert len(missing_example_pages) == 0
assert len(missing_targets) == 0
assert audit_alignment == "validated"

CHECKPOINT_METRICS = {
    "topics_in_manifest": float(len(topics)),
    "missing_example_pages": float(len(missing_example_pages)),
    "toc_targets": float(len(targets)),
    "missing_targets": float(len(missing_targets)),
}
CHECKPOINT_LIMITS = {
    "topics_in_manifest": (1.0, 5000.0),
    "missing_example_pages": (0.0, 0.0),
    "toc_targets": (1.0, 5000.0),
    "missing_targets": (0.0, 0.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
